# 05 — Social Media Posting (Phase 1b)

Validate direct posting to Mastodon and Bluesky APIs.

**Prerequisites:**
- Mastodon: Create app at mastodon.social → Settings → Development
  - Scopes: `read:accounts`, `read:statuses`, `write:statuses`
  - Add `MASTODON_ACCESS_TOKEN` to `.env`
- Bluesky: Generate app password at bsky.app → Settings → App Passwords
  - Add `BLUESKY_APP_PASSWORD` to `.env`

> **Do not run this notebook until auth tokens are configured.**

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env')

MASTODON_ACCESS_TOKEN = os.getenv('MASTODON_ACCESS_TOKEN')
BLUESKY_APP_PASSWORD = os.getenv('BLUESKY_APP_PASSWORD')
BLUESKY_HANDLE = os.getenv('BLUESKY_HANDLE', 'fretchen.eu')
MASTODON_INSTANCE = os.getenv('MASTODON_INSTANCE', 'https://mastodon.social')

print(f'Mastodon token: {"yes" if MASTODON_ACCESS_TOKEN else "NOT SET"}')
print(f'Bluesky password: {"yes" if BLUESKY_APP_PASSWORD else "NOT SET"}')
print(f'Mastodon instance: {MASTODON_INSTANCE}')
print(f'Bluesky handle: {BLUESKY_HANDLE}')

Mastodon token: yes
Bluesky password: yes
Mastodon instance: https://mastodon.social
Bluesky handle: fretchen.eu


## 1. Mastodon — Verify Credentials

In [2]:
import httpx

if not MASTODON_ACCESS_TOKEN:
    print('⏭️ Skipping Mastodon — no token configured.')
else:
    client = httpx.Client(
        base_url=MASTODON_INSTANCE,
        headers={'Authorization': f'Bearer {MASTODON_ACCESS_TOKEN}'},
    )
    
    account = client.get('/api/v1/accounts/verify_credentials').json()
    print(f'Logged in as: @{account["acct"]}')
    print(f'Followers: {account["followers_count"]}')
    print(f'Following: {account["following_count"]}')
    print(f'Posts: {account["statuses_count"]}')

Logged in as: @fretchen
Followers: 3
Following: 21
Posts: 19


## 2. Mastodon — Post a Test Status

⚠️ This will actually post to Mastodon. Use `visibility: 'direct'` for testing
(only visible to you).

In [3]:
if not MASTODON_ACCESS_TOKEN:
    print('⏭️ Skipping.')
else:
    # Post with 'direct' visibility for testing (only you can see it)
    test_post = client.post('/api/v1/statuses', json={
        'status': 'Growth Agent test post — please ignore. 🔬',
        'visibility': 'direct',  # Change to 'public' for real posts
    }).json()
    
    print(f'Posted! ID: {test_post["id"]}')
    print(f'URL: {test_post["url"]}')
    
    # Clean up: delete the test post
    client.delete(f'/api/v1/statuses/{test_post["id"]}')
    print('Deleted test post.')

Posted! ID: 116397563830591912
URL: https://mastodon.social/@fretchen/116397563830591912
Deleted test post.


## 3. Bluesky — Authenticate

In [4]:
if not BLUESKY_APP_PASSWORD:
    print('⏭️ Skipping Bluesky — no app password configured.')
else:
    bsky_client = httpx.Client(base_url='https://bsky.social')
    
    # Create session
    session = bsky_client.post('/xrpc/com.atproto.server.createSession', json={
        'identifier': BLUESKY_HANDLE,
        'password': BLUESKY_APP_PASSWORD,
    }).json()
    
    bsky_did = session['did']
    bsky_token = session['accessJwt']
    
    print(f'Authenticated as: {session["handle"]}')
    print(f'DID: {bsky_did}')
    
    # Get profile
    profile = bsky_client.get(
        '/xrpc/app.bsky.actor.getProfile',
        params={'actor': bsky_did},
        headers={'Authorization': f'Bearer {bsky_token}'},
    ).json()
    
    print(f'Followers: {profile.get("followersCount", 0)}')
    print(f'Following: {profile.get("followsCount", 0)}')
    print(f'Posts: {profile.get("postsCount", 0)}')

Authenticated as: fretchen.eu
DID: did:plc:t3362qxvovwfdx6fhw4ux642
Followers: 8
Following: 33
Posts: 44


## 4. Bluesky — Post a Test Status

⚠️ Bluesky has no 'direct' visibility. This creates a real post.
We immediately delete it after.

In [5]:
from datetime import datetime, timezone

if not BLUESKY_APP_PASSWORD:
    print('⏭️ Skipping.')
else:
    # Create a post
    post_result = bsky_client.post(
        '/xrpc/com.atproto.repo.createRecord',
        headers={'Authorization': f'Bearer {bsky_token}'},
        json={
            'repo': bsky_did,
            'collection': 'app.bsky.feed.post',
            'record': {
                '$type': 'app.bsky.feed.post',
                'text': 'Growth Agent test post — please ignore.',
                'createdAt': datetime.now(timezone.utc).isoformat(),
            },
        },
    ).json()
    
    rkey = post_result['uri'].split('/')[-1]
    print(f'Posted! URI: {post_result["uri"]}')
    
    # Delete the test post
    bsky_client.post(
        '/xrpc/com.atproto.repo.deleteRecord',
        headers={'Authorization': f'Bearer {bsky_token}'},
        json={
            'repo': bsky_did,
            'collection': 'app.bsky.feed.post',
            'rkey': rkey,
        },
    )
    print('Deleted test post.')

Posted! URI: at://did:plc:t3362qxvovwfdx6fhw4ux642/app.bsky.feed.post/3mjewvvmtnr2p
Deleted test post.


## 5. Publish Draft from Content Queue

Load an approved draft and post it for real.

In [6]:
from agent.storage import LocalStorage

store = LocalStorage(base_dir='../state')
queue_data = store.read('content_queue.json')

if queue_data:
    drafts = queue_data.get('drafts', [])
    print(f'Pending drafts: {len(drafts)}')
    for d in drafts:
        print(f'  - {d["id"]} | {d["channel"]} ({d["language"]}) | {len(d["content"])} chars')
        print(f'    {d["content"][:100]}...')
else:
    print('No content queue — run notebook 03 first.')

Pending drafts: 3
  - draft_cb352481 | mastodon (en) | 297 chars
    Can decentralized platforms truly scale? One key insight from our latest article is that independent...
  - draft_9a82a5a4 | bluesky (en) | 173 chars
    "Unlocking growth on Optimism and Base: get the inside scoop from an independent x402 facilitator ht...
  - draft_8e4b825f | mastodon (de) | 311 chars
    Kann ein unabhängiger Facilitator wirklich den Erfolg von Optimism und Base entscheiden? Ein interes...


In [ ]:
print('Done — Social posting validation complete.')

Done — Social posting validation complete.

Next steps:
1. Create Mastodon app and add MASTODON_ACCESS_TOKEN to .env
2. Generate Bluesky app password and add BLUESKY_APP_PASSWORD to .env
3. Re-run this notebook to test posting
